Part 3: More Queries
In addition to the above tasks, your manager also wants to know the following additional queries:
1) Which neighborhood has the most trips that start in it?
2) How many trips pass through the Upper East Side but do not start or end in that neighborhood?
3) What is the area of each neighborhood in square miles? Sort by those with the largest area.

In [ ]:
import duckdb

conn = duckdb.connect("taxi_analysis.duckdb")
conn.execute("INSTALL spatial;")
conn.execute("LOAD spatial;")

conn.execute("SHOW TABLES").fetchdf()

,name
0,classified_trips
1,neighborhoods_2263
2,part1_trip_speeds
3,part2_neighborhood_speeds
4,taxi_trips
5,tourist_spots
6,trip_lines_2263
7,trip_speeds_2263
8,trips_near_tourist
9,trips_with_distance


In [ ]:
#1: Which neighborhood has the most trips that start in it?

part3q1 = conn.execute("""
    WITH pickup_points AS (
        SELECT
            id,
            ST_Transform(
                ST_Point(pickup_longitude, pickup_latitude),
                'EPSG:4326',
                'EPSG:2263',
                always_xy := true
            ) AS pickup_geom
        FROM taxi_trips
        WHERE
            pickup_longitude IS NOT NULL
            AND pickup_latitude IS NOT NULL
    )
    SELECT
        n.neighborhood,
        COUNT(DISTINCT p.id) AS total_trips
    FROM pickup_points AS p
    INNER JOIN neighborhoods_2263 AS n
        ON ST_Within(p.pickup_geom, n.geom)
    GROUP BY n.neighborhood
    ORDER BY total_trips DESC
    LIMIT 1
""").fetchdf()

part3q1.to_csv("part3q1.csv", index=False)


In [ ]:
# 2) How many trips pass through the Upper East Side but do not start or end in that neighborhood?

part3q2 = conn.execute("""
    WITH upper_east_side AS (
        SELECT geom
        FROM neighborhoods_2263
        WHERE neighborhood = 'Upper East Side'
    ),
    pickup_dropoff_points AS (
        SELECT
            id,
            ST_Transform(
                ST_Point(pickup_longitude, pickup_latitude),
                'EPSG:4326',
                'EPSG:2263',
                always_xy := true
            ) AS pickup_geom,
            ST_Transform(
                ST_Point(dropoff_longitude, dropoff_latitude),
                'EPSG:4326',
                'EPSG:2263',
                always_xy := true
            ) AS dropoff_geom
        FROM taxi_trips
        WHERE
            pickup_longitude IS NOT NULL
            AND pickup_latitude IS NOT NULL
            AND dropoff_longitude IS NOT NULL
            AND dropoff_latitude IS NOT NULL
    )
    SELECT
        COUNT(DISTINCT t.id) AS "number of trips"
    FROM trip_lines_2263 AS t
    INNER JOIN upper_east_side AS u
        ON ST_Intersects(t.geom, u.geom)
    INNER JOIN pickup_dropoff_points AS p
        ON p.id = t.id
    WHERE
        NOT ST_Within(p.pickup_geom, u.geom)
        AND NOT ST_Within(p.dropoff_geom, u.geom)
""").fetchdf()

part3q2.to_csv("part3q2.csv", index=False)

part3q2


,number of trips
0,59361


In [ ]:
# 3) What is the area of each neighborhood in square miles? Sort by those with the largest area.
# Convert to sq miles

part3q3 = conn.execute("""
    SELECT 
        neighborhood,
        ST_Area(geom) / (5280* 5280) as area
    FROM neighborhoods_2263
    ORDER BY area DESC
""").fetchdf()

part3q3.to_csv("part3q3.csv",index=False)

### Reminder Spatial Extension

ST_Point
Creates pickup/dropoff/tourist points.

ST_Transform
Changes coordinate systems.

ST_Distance
Measures distance between two geometries.

ST_MakeLine
Turns pickup/dropoff points into a trip line.

ST_Length
Measures length of a trip line.

ST_Intersects
Checks whether a trip line passes through a neighborhood.

ST_Within
Checks whether a point starts/ends inside a neighborhood.

ST_Area
Calculates neighborhood area.

ST_AsGeoJSON
Converts geometry for Folium map display.

ST_Read
Loads the shapefile into DuckDB.

**always_xy := true -> tells DuckDB: Always interpret coordinates as x, y.